# RAGulate Tutorial: Inference-Only Regulatory Prediction

This notebook demonstrates how to use **RAGulate** for **post-hoc, literature-grounded regulatory inference**
using a **precomputed document store** and **your own Mistral (mistralai) language model**.

## What this notebook does
- Downloads the required RAGulate document store from Zenodo
- Loads a Mistral LLM via Hugging Face
- Runs regulatory edge predictions (TF → target) in a context-specific manner
- Returns a confidence score, LLM justification, and supporting PMIDs

## What this notebook does not do
- No training
- No indexing
- No document preprocessing
- No database construction

This notebook is inference-only.


## 1. Environment requirements

Before running this notebook, ensure:

- Python ≥ 3.9
- Internet access
- Access to a Mistral model on Hugging Face
- A valid `HUGGINGFACE_HUB_TOKEN` set in your environment if required

Example:

```bash
export HUGGINGFACE_HUB_TOKEN=hf_...


In [1]:
from pathlib import Path

## 2. Create workspace and define paths

In [2]:
DATA_DIR = Path("ragulate_data")
DATA_DIR.mkdir(exist_ok=True)

PICKLE_PATH = DATA_DIR / "collectri_docs.pkl"

## 3. Download the RAGulate document store

RAGulate relies on a **precomputed document store** derived from curated regulatory
resources and the biomedical literature.

The file is hosted on **Zenodo** and is required for reproducibility.

- File: `collectri_docs.pkl`
- Source: Zenodo
- Downloaded automatically by this notebook


In [3]:
import requests
from pathlib import Path

DATA_DIR = Path("ragulate_data")
CACHE_DIR = DATA_DIR / "pubmed_cache"
DATA_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

# Local paths
SQLITE_PATH = DATA_DIR / "pubmed_cache.sqlite"
NPZ_PATH = CACHE_DIR / "embeddings-sentence-transformers__all-MiniLM-L6-v2.npz"
META_PATH = CACHE_DIR / "embeddings-sentence-transformers__all-MiniLM-L6-v2.meta.json"
HGNC_PATH = DATA_DIR / "hgnc_complete_set.txt"

# Zenodo URLs
SQLITE_URL = "https://zenodo.org/records/18511315/files/pubmed_cache.sqlite?download=1"
NPZ_URL = "https://zenodo.org/records/18511315/files/embeddings-sentence-transformers__all-MiniLM-L6-v2.npz?download=1"
META_URL = "https://zenodo.org/records/18511315/files/embeddings-sentence-transformers__all-MiniLM-L6-v2.meta.json?download=1"
HGNC_URL = "https://zenodo.org/records/18511315/files/hgnc_complete_set.txt?download=1"


def download_if_missing(url: str, path: Path, label: str) -> None:
    if path.exists() and path.stat().st_size > 0:
        print(f"{label} already exists.")
        return

    print(f"Downloading {label} from Zenodo...")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        tmp = path.with_suffix(path.suffix + ".tmp")
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        tmp.replace(path)
    print(f"{label} download complete.")


download_if_missing(SQLITE_URL, SQLITE_PATH, "pubmed_cache.sqlite")
download_if_missing(NPZ_URL, NPZ_PATH, "embeddings .npz")
download_if_missing(META_URL, META_PATH, "embeddings meta.json")
download_if_missing(HGNC_URL, HGNC_PATH, "hgnc_complete_set.txt")

pubmed_cache.sqlite already exists.
embeddings .npz already exists.
embeddings meta.json already exists.
hgnc_complete_set.txt already exists.


In [4]:
from ragulate_bio import config

# point to your downloaded files
config.PUBMED_SQLITE = str(DATA_DIR / "pubmed_cache.sqlite")
config.PUBMED_CACHE  = str(CACHE_DIR)
config.HGNC_COMPLETE_SET_FILE = str(DATA_DIR / "hgnc_complete_set.txt")

# recompute embedding cache paths based on PUBMED_CACHE
config.EMB_CACHE_BASENAME = f"embeddings-{config.EMBED_MODEL_NAME.replace('/', '__')}"
config.EMB_CACHE_NPZ = str(CACHE_DIR / (config.EMB_CACHE_BASENAME + ".npz"))
config.EMB_CACHE_META = str(CACHE_DIR / (config.EMB_CACHE_BASENAME + ".meta.json"))

print("SQLITE:", config.PUBMED_SQLITE)
print("CACHE :", config.PUBMED_CACHE)
print("NPZ   :", config.EMB_CACHE_NPZ)
print("META  :", config.EMB_CACHE_META)

# ONLY NOW import the rest of ragulate
import ragulate_bio.pipeline as pipeline

SQLITE: ragulate_data/pubmed_cache.sqlite
CACHE : ragulate_data/pubmed_cache
NPZ   : ragulate_data/pubmed_cache/embeddings-sentence-transformers__all-MiniLM-L6-v2.npz
META  : ragulate_data/pubmed_cache/embeddings-sentence-transformers__all-MiniLM-L6-v2.meta.json


## 4. Configure RAGulate to use the downloaded resources

RAGulate uses a centralized configuration file.
Here we explicitly point the pipeline to the downloaded document store to ensure full reproducibility.

In [5]:
# Point RAGulate to the minimal resources
ragulate.config.PUBMED_SQLITE = str(SQLITE_PATH)
ragulate.config.PUBMED_CACHE = str(CACHE_DIR)
ragulate.config.HGNC_COMPLETE_SET_FILE = str(HGNC_PATH)

NameError: name 'ragulate' is not defined

## 5. Load the Mistral language model

RAGulate supports large language models for final regulatory assessment.

In this tutorial, we use a **Mistral (mistralai)** model via Hugging Face.

Default model:
- `mistralai/Mistral-7B-Instruct-v0.2`

In [5]:
tokenizer, model = ragulate.llm_models.get_mistral()

[init] Mistral -> mistralai/Mistral-7B-Instruct-v0.2


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


[ready] Mistral initialized


## 6. Optional sanity check

This step verifies that the Mistral model is correctly loaded and responding.


In [6]:
print(
    ragulate.llm_models._llm_generate(
        tokenizer,
        model,
        "Answer with yes or no: Does TP53 regulate CDKN1A?",
        max_new_tokens=32,
    )
)

Answer with yes or no: Does TP53 regulate CDKN1A?

Yes, TP53 regulates CDKN1A. TP53 is a transcription factor that can bind to the CDK


## 7. Define regulatory queries

Each row corresponds to a putative regulatory edge defined by:
- Transcription factor
- Target gene
- Biological context (cell type, tissue, or condition)


In [7]:
import pandas as pd

edges = pd.DataFrame([
    {"tf": "PAX5",  "target": "CD19",  "context": "naive B cell"},
    {"tf": "TBX21", "target": "GZMB",  "context": "NK cell"},
    {"tf": "SPI1",  "target": "CSF1R", "context": "monocyte"},
])

## 8. Run RAGulate inference

RAGulate combines:
- Hybrid retrieval (BM25 + embedding similarity)
- Alias-aware querying
- LLM-based regulatory assessment
- Literature-backed evidence aggregation


In [8]:
import ragulate_bio.retrieval as R

R._BM25_INDEX = None
R._BM25_TOKENS = None
R._BM25_PMIDS = None
if hasattr(R, "_BM25_SOURCE_PATH"):
    R._BM25_SOURCE_PATH = None


In [9]:
results = ragulate.pipeline.run_ragulate_inference(
    edges_df=edges,
    method_type="hybrid",
    encoder_name="minilm",
    top_k_docs=20,
    use_llm=True,
    llm_name=ragulate.config.MISTRAL_MODEL_NAME,
    classify=True,
    use_aliases=True,
)

[init] Sentence embedder -> sentence-transformers/all-MiniLM-L6-v2
[ready] Sentence embedder initialized
[warn] could not save embedding cache: [Errno 2] No such file or directory: './pubmed_cache/embeddings-sentence-transformers__all-MiniLM-L6-v2.npz'


RuntimeError: MiniLM global corpus was not initialised correctly (0 docs)

## 9. Inspect results


In [ ]:
results[
    [
        "tf",
        "target",
        "context",
        "score",
        "llm_summary",
        "retrieved_pmids",
    ]
]


## 10. Save predictions


To save in tabular `.csv` format:

In [ ]:
results.to_csv("ragulate_predictions.csv", index=False)

To save in web `.html` format:

In [ ]:
results.to_html("ragulate_predictions.html", index=False)

## Summary

You have now:
- Downloaded all required RAGulate resources automatically
- Loaded your own Mistral LLM
- Queried regulatory hypotheses
- Obtained literature-grounded predictions with confidence scores
